#### Synapse Connection

In [17]:
import pyodbc


server = 'syn-rt-prd-unity.sql.azuresynapse.net'
database = 'syndwrtprdunity'
username = 'byambadorjme@riotinto.com'  
driver = '{ODBC Driver 17 for SQL Server}'

connection_string = f"""
    DRIVER={driver};
    SERVER={server};
    DATABASE={database};
    UID={username};
    Authentication=ActiveDirectoryInteractive;
    Encrypt=yes;
    TrustServerCertificate=no;
    Connection Timeout=30;
"""


connection = pyodbc.connect(connection_string)

In [ ]:
ltlh
omd

In [ ]:
Cause, 
DateDebut,	
DateFin,
Description,
DureeSec,
IDCodeArret,
IDEmploye,
IDEquipement,
IDOperation,
IDSource,
Localisation

In [18]:
synapse_schema_name = 'HSTG_V'
synapse_table_name = 'VW_HSTG_UDLGENSD03_DBO_HSP_SUIVIETATEQUIPEMENTROWNUMBER_VIEW'

In [19]:
from databricks import sql

catalog_name = '_ogr_azr_syd'
schema_name = 'gensd03_sicom'
table_name = 'dbo_hsp_suivietatequipementrownumber_view'

connection = sql.connect(
    server_hostname = "adb-2439498039309203.3.azuredatabricks.net",
    http_path = "/sql/1.0/warehouses/e7d134c4348ac8b5",
    access_token = "os.environ["DATABRICKS_TOKEN"]"
)

In [20]:
# Compare sample data (first 10 rows)
print("=" * 80)
print("SAMPLE DATA COMPARISON (First 5 rows)")
print("=" * 80)
print("\nSynapse Sample:")
print(synapse_compare_df.head())
print("\nDatabricks Sample:")
print(databricks_compare_df.head())
print()

SAMPLE DATA COMPARISON (First 5 rows)

Synapse Sample:


NameError: name 'synapse_compare_df' is not defined

In [ ]:
# Compare data types
print("=" * 80)
print("DATA TYPE COMPARISON")
print("=" * 80)
dtype_comparison = pd.DataFrame({
    'Column': common_cols_list,
    'Synapse_dtype': [str(synapse_compare_df[col].dtype) for col in common_cols_list],
    'Databricks_dtype': [str(databricks_compare_df[col].dtype) for col in common_cols_list]
})
dtype_comparison['Match'] = dtype_comparison['Synapse_dtype'] == dtype_comparison['Databricks_dtype']

print(dtype_comparison)
print(f"\nData type mismatches: {(~dtype_comparison['Match']).sum()}")
print()

In [ ]:
# Compare row counts
print("=" * 80)
print("ROW COUNT COMPARISON")
print("=" * 80)
print(f"Synapse row count: {len(synapse_compare_df)}")
print(f"Databricks row count: {len(databricks_compare_df)}")
print(f"Difference: {abs(len(synapse_compare_df) - len(databricks_compare_df))}")
print()

In [ ]:
# Normalize column names for comparison (to lowercase)
synapse_filtered_df.columns = [col.lower() for col in synapse_filtered_df.columns]
databricks_filtered_df.columns = [col.lower() for col in databricks_filtered_df.columns]

# Select only common columns from both dataframes
common_cols_list = sorted(list(common_columns))
synapse_compare_df = synapse_filtered_df[common_cols_list].copy()
databricks_compare_df = databricks_filtered_df[common_cols_list].copy()

print(f"Comparing {len(common_cols_list)} common columns")
print(f"Columns to compare: {common_cols_list}")

In [ ]:
# Compare Column Names (case-insensitive)
synapse_cols_lower = set([col.lower() for col in synapse_filtered_df.columns])
databricks_cols_lower = set([col.lower() for col in databricks_filtered_df.columns])

only_in_synapse = synapse_cols_lower - databricks_cols_lower
only_in_databricks = databricks_cols_lower - synapse_cols_lower
common_columns = synapse_cols_lower.intersection(databricks_cols_lower)

print("=" * 80)
print("COLUMN COMPARISON")
print("=" * 80)
print(f"Common columns: {len(common_columns)}")
print(f"Columns only in Synapse: {len(only_in_synapse)}")
if only_in_synapse:
    print(f"  → {only_in_synapse}")
print(f"Columns only in Databricks: {len(only_in_databricks)}")
if only_in_databricks:
    print(f"  → {only_in_databricks}")
print()

In [ ]:
# Filter out columns starting with 'omd' from Synapse and 'rtlh' from Databricks
synapse_filtered_cols = [col for col in synapse_df.columns if not col.lower().startswith('omd')]
databricks_filtered_cols = [col for col in databricks_df.columns if not col.lower().startswith('rtlh')]

synapse_filtered_df = synapse_df[synapse_filtered_cols].copy()
databricks_filtered_df = databricks_df[databricks_filtered_cols].copy()

print(f"Synapse after filtering: {len(synapse_filtered_df.columns)} columns")
print(f"Synapse filtered columns: {list(synapse_filtered_df.columns)}")
print(f"\nDatabricks after filtering: {len(databricks_filtered_df.columns)} columns")
print(f"Databricks filtered columns: {list(databricks_filtered_df.columns)}")

In [ ]:
# Query Databricks Table
print("Connecting to Databricks...")
databricks_conn = sql.connect(
    server_hostname=databricks_server_hostname,
    http_path=databricks_http_path,
    access_token=databricks_access_token
)

databricks_query = f"SELECT * FROM {databricks_catalog_name}.{databricks_schema_name}.{databricks_table_name} LIMIT 10000"
cursor = databricks_conn.cursor()
cursor.execute(databricks_query)

# Fetch results into DataFrame
databricks_data = cursor.fetchall()
databricks_columns = [desc[0] for desc in cursor.description]
databricks_df = pd.DataFrame(databricks_data, columns=databricks_columns)

cursor.close()
databricks_conn.close()

print(f"Databricks data loaded: {len(databricks_df)} rows, {len(databricks_df.columns)} columns")
print(f"Databricks columns: {list(databricks_df.columns)}")

In [ ]:
# Query Synapse Table
print("Connecting to Synapse...")
synapse_conn = pyodbc.connect(synapse_connection_string)
synapse_query = f"SELECT TOP 10000 * FROM {synapse_schema_name}.{synapse_table_name}"
synapse_df = pd.read_sql(synapse_query, synapse_conn)
synapse_conn.close()

print(f"Synapse data loaded: {len(synapse_df)} rows, {len(synapse_df.columns)} columns")
print(f"Synapse columns: {list(synapse_df.columns)}")

In [ ]:
import pandas as pd
import pyodbc
from databricks import sql

# Synapse Connection
synapse_server = 'syn-rt-prd-unity.sql.azuresynapse.net'
synapse_database = 'syndwrtprdunity'
synapse_username = 'byambadorjme@riotinto.com'  
synapse_driver = '{ODBC Driver 17 for SQL Server}'
synapse_schema_name = 'HSTG_V'
synapse_table_name = 'VW_HSTG_UDLGENSD03_DBO_HSP_SUIVIETATEQUIPEMENTROWNUMBER_VIEW'

synapse_connection_string = f"""
    DRIVER={synapse_driver};
    SERVER={synapse_server};
    DATABASE={synapse_database};
    UID={synapse_username};
    Authentication=ActiveDirectoryInteractive;
    Encrypt=yes;
    TrustServerCertificate=no;
    Connection Timeout=30;
"""

# Databricks Connection
databricks_access_token = os.environ["DATABRICKS_TOKEN"]
databricks_server_hostname = 'adb-2439498039309203.3.azuredatabricks.net'
databricks_http_path = '/sql/1.0/warehouses/e7d134c4348ac8b5'
databricks_catalog_name = '_ogr_azr_syd'
databricks_schema_name = 'gensd03_sicom'
databricks_table_name = 'dbo_hsp_suivietatequipementrownumber_view'

print("Configuration loaded successfully")

#### Data Comparison: Synapse vs Databricks